필요 라이브러리 설치

In [1]:
%pip install -q ultralytics==8.4.63 kagglehub==1.0.3 pyyaml

Kaggle 데이터셋 다운로드

In [2]:
from pathlib import Path
import kagglehub

SOURCE = Path(
    kagglehub.dataset_download(
        "a7madmostafa/bdd100k-yolo/versions/4"
    )
)

print("데이터셋 위치:", SOURCE)
print("폴더 내용:", [p.name for p in SOURCE.iterdir()])

100%|██████████| 5.33G/5.33G [00:59<00:00, 96.1MB/s]

Extracting files...


데이터셋 위치: /root/.cache/kagglehub/datasets/a7madmostafa/bdd100k-yolo/versions/4
폴더 내용: ['data.yaml', 'val', 'train', 'test']


사람, 차량, 표지판 3개 부류 라벨 준비

원본의 학습, 검증 분할 유지.

학습 3000장, 검증 500장으로 진행

In [3]:
import random
import tempfile
from collections import Counter
import yaml

# 해당 Kaggle 데이터셋 버전 4의 클래스 번호 확인
original = yaml.safe_load(
    (SOURCE / "data.yaml").read_text(encoding="utf-8")
)

old_names = original["names"]
if isinstance(old_names, dict):
    old_names = [old_names[k] for k in sorted(old_names, key=int)]

assert [old_names[i] for i in [0, 2, 3, 4, 8]] == [
    "person", "car", "bus", "truck", "traffic sign"
], "데이터셋의 클래스 순서를 확인하세요."

# 원본 ID → 새 ID
CLASS_MAP = {
    0: 0,  # person → person
    2: 1,  # car → vehicle
    3: 1,  # bus → vehicle
    4: 1,  # truck → vehicle
    8: 2,  # traffic sign → traffic_sign
}

NAMES = ["person", "vehicle", "traffic_sign"]
LIMITS = {"train": 3000, "val": 500}

# 다시 실행해도 기존 준비 데이터를 섞지 않도록 새 폴더 사용
DATA = Path(tempfile.mkdtemp(prefix="road3_", dir="/content"))


def prepare_split(split, limit):
    image_root = SOURCE / split / "images"
    label_root = SOURCE / split / "labels"

    images = sorted(
        p for p in image_root.rglob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    assert images, f"이미지 폴더를 확인하세요: {image_root}"

    # 대응 라벨 파일이 있는 이미지를 사용
    images = [
        p for p in images
        if (label_root / p.relative_to(image_root).with_suffix(".txt")).is_file()
    ]

    random.Random(42).shuffle(images)
    if limit is not None:
        images = images[:limit]

    counts = Counter()

    for image in images:
        relative = image.relative_to(image_root)
        source_label = label_root / relative.with_suffix(".txt")

        converted = []
        present = set()

        for line in source_label.read_text().splitlines():
            if not line.strip():
                continue

            fields = line.split()
            assert len(fields) == 5, f"경계상자 라벨 형식 확인: {source_label}"

            old_id = int(fields[0])
            if old_id in CLASS_MAP:
                new_id = CLASS_MAP[old_id]
                converted.append(
                    f"{new_id} {' '.join(fields[1:])}"
                )
                present.add(new_id)

        output_image = DATA / split / "images" / relative
        output_label = (
            DATA / split / "labels" / relative.with_suffix(".txt")
        )

        output_image.parent.mkdir(parents=True, exist_ok=True)
        output_label.parent.mkdir(parents=True, exist_ok=True)

        # 이미지는 복제하지 않고 연결하고, 변환 라벨만 새로 저장
        output_image.symlink_to(image.resolve())
        output_label.write_text("\n".join(converted), encoding="utf-8")

        counts.update(present)

    assert all(counts[i] > 0 for i in range(3)), \
        f"{split}에 세 부류가 모두 있는지 확인하세요."

    print(
        split,
        "이미지 수:", len(images),
        "부류별 포함 이미지 수:",
        {NAMES[i]: counts[i] for i in range(3)}
    )

    return counts


train_counts = prepare_split("train", LIMITS["train"])
val_counts = prepare_split("val", LIMITS["val"])

assert all(
    train_counts[i] + val_counts[i] >= 100
    for i in range(3)
), "부류별 이미지가 부족합니다. LIMITS를 늘리세요."

DATA_YAML = DATA / "data.yaml"

DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(DATA),
            "train": "train/images",
            "val": "val/images",
            "nc": 3,
            "names": NAMES,
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)

print("학습 설정 파일:", DATA_YAML)

train 이미지 수: 3000 부류별 포함 이미지 수: {'person': 956, 'vehicle': 2979, 'traffic_sign': 2465}
val 이미지 수: 500 부류별 포함 이미지 수: {'person': 164, 'vehicle': 497, 'traffic_sign': 424}
학습 설정 파일: /content/road3_68evza9i/data.yaml


YOLO 학습

In [4]:
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), \
    "Colab에서 GPU 런타임을 선택하세요."

model = YOLO("yolov8s.pt")

model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    seed=42,
    patience=10,
    project="/content/road_runs",
    name="person_vehicle_sign",
)

print("학습된 모델:", model.trainer.best)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.216 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.63 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road3_68evza9i/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipu

학습한 모델 확인

In [5]:
%pip install -q "lap>=0.5.12"

from pathlib import Path
from ultralytics import YOLO

# 앞에서 학습한 모델의 가중치
BEST_PT = Path(model.trainer.best)

assert BEST_PT.is_file(), "학습된 best.pt 경로를 확인하세요."

names = YOLO(str(BEST_PT)).names
print("학습된 모델:", BEST_PT)
print("학습된 부류:", names)

assert set(names.values()) == {"person", "vehicle", "traffic_sign"}

학습된 모델: /content/road_runs/person_vehicle_sign/weights/best.pt
학습된 부류: {0: 'person', 1: 'vehicle', 2: 'traffic_sign'}


학습 부류 추적할 도로 영상 업로드

In [6]:
from google.colab import files

uploaded = files.upload()  # 영상 한 개 선택

assert len(uploaded) == 1, "영상 한 개만 선택하세요."

INPUT_VIDEO = Path(next(iter(uploaded))).resolve()
del uploaded

print("입력 영상:", INPUT_VIDEO)

Saving Driving video.mp4 to Driving video.mp4
입력 영상: /content/Driving video.mp4


객체 감지 및 추적 실행

In [8]:
from pathlib import Path
import tempfile
import cv2
import subprocess
from ultralytics import YOLO

# 업로드한 영상
CLIP = Path("/content/Driving video.mp4")

# 앞에서 설정한 학습 가중치 경로 사용
BEST_PT = Path(BEST_PT)

assert CLIP.is_file(), f"영상 파일을 찾을 수 없습니다: {CLIP}"
assert BEST_PT.is_file(), f"학습 모델을 찾을 수 없습니다: {BEST_PT}"

# 이번 실행의 결과 폴더
WORK = Path(
    tempfile.mkdtemp(prefix="road_tracking_", dir="/content")
)

# 새 모델을 불러와 추적 ID 초기화
detector = YOLO(str(BEST_PT))

print("입력 영상:", CLIP)
print("학습된 부류:", detector.names)

processed = 0

# 영상 전체에 객체 감지·추적 적용
for result in detector.track(
    source=str(CLIP),
    tracker="botsort.yaml",
    persist=True,
    conf=0.10,
    imgsz=640,
    device=0,
    save=True,
    stream=True,
    show=False,
    verbose=False,
    project=str(WORK / "tracking"),
    name="result",
    exist_ok=True,
):
    processed += 1

    if processed % 30 == 0:
        print(f"{processed}프레임 처리 중...")

assert processed > 0, "영상이 처리되지 않았습니다."

# 저장된 결과 영상 찾기
saved_dir = Path(detector.predictor.save_dir)

videos = [
    p for p in saved_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {".avi", ".mp4"}
]

assert len(videos) == 1, \
    f"저장된 결과 영상을 확인하세요: {saved_dir}"

TRACKED_VIDEO = videos[0]

print("추적 완료!")
print("처리한 프레임 수:", processed)
print("결과 영상:", TRACKED_VIDEO)

입력 영상: /content/Driving video.mp4
학습된 부류: {0: 'person', 1: 'vehicle', 2: 'traffic_sign'}
30프레임 처리 중...
60프레임 처리 중...
90프레임 처리 중...
120프레임 처리 중...
150프레임 처리 중...
180프레임 처리 중...
210프레임 처리 중...
240프레임 처리 중...
270프레임 처리 중...
Results saved to /content/road_tracking_fabqf4y1/tracking/result
추적 완료!
처리한 프레임 수: 299
결과 영상: /content/road_tracking_fabqf4y1/tracking/result/Driving video.avi


추적 영상 MP4 변환. 재생 및 다운로드

In [10]:
from IPython.display import Video, display

OUTPUT_VIDEO = WORK / "tracking_result.mp4"

subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(TRACKED_VIDEO),
    "-an",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    str(OUTPUT_VIDEO),
], check=True)

# 결과 영상 길이 확인
cap = cv2.VideoCapture(str(OUTPUT_VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS)
decoded = 0

while True:
    ok, _ = cap.read()
    if not ok:
        break
    decoded += 1

cap.release()

assert decoded == processed, "처리한 프레임 수와 저장된 프레임 수가 다릅니다."
print(f"결과: {decoded}프레임 / {fps:.1f}fps / {decoded / fps:.1f}초")

display(Video(str(OUTPUT_VIDEO), embed=True))
files.download(str(OUTPUT_VIDEO))

결과: 299프레임 / 29.0fps / 10.3초


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

학습 모델(best.pt) 다운로드

In [11]:
from google.colab import files

files.download(
    "/content/road_runs/person_vehicle_sign/weights/best.pt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>